# Support Data Insight Analysis for Loan Prediction System

## Initial Imports

In [5]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

# Load environment variables
from helper import load_env
load_env()

# multi_agent_data_eng_report_fixed.py
# Combined workflow with fixes for typing/schema and final markdown assembly
import os
import json
import warnings
from pprint import pprint
from typing import List, Dict, Any
import pandas as pd
from pydantic import BaseModel
from crewai import Agent, Task, Crew
from crewai_tools import FileReadTool, SerperDevTool, ScrapeWebsiteTool
from IPython.display import display, Markdown

## Loading Tasks and Agents YAML files

In [ ]:
# # Define file paths for YAML configurations
# files = {
#     'agents': 'config/agents.yaml',
#     'tasks': 'config/tasks.yaml'
# }

# # Load configurations from YAML files
# configs = {}
# for config_type, file_path in files.items():
#     with open(file_path, 'r') as file:
#         configs[config_type] = yaml.safe_load(file)

# # Assign loaded configurations to specific variables
# agents_config = configs['agents']
# tasks_config = configs['tasks']

## Kicking off Crew

In [6]:
# -------------------------
# Config & paths
# -------------------------
RAW_CSV_PATH = "./Training_Data.csv"   # <-- chỉnh đường dẫn nếu cần
ARTIFACTS_DIR = "artifacts"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

CSV_SUMMARY_PATH = os.path.join(ARTIFACTS_DIR, "csv_summary.json")
CREW_RAW_JSON = os.path.join(ARTIFACTS_DIR, "crew_result_raw.json")
FINAL_REPORT_MD = os.path.join(ARTIFACTS_DIR, "final_report.md")

# -------------------------
# Create lightweight CSV summary (safe for LLM prompts)
# -------------------------
def create_csv_summary(csv_path: str, sample_rows: int = 500) -> Dict[str, Any]:
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"CSV not found: {csv_path}")
    df = pd.read_csv(csv_path, nrows=sample_rows)
    summary: Dict[str, Any] = {
        "sample_rows": len(df),
        "num_columns": len(df.columns),
        "columns": {},
    }
    for col in df.columns:
        nonnull = df[col].dropna()
        sample_values = nonnull.unique()[:5].tolist() if len(nonnull) > 0 else []
        summary["columns"][col] = {
            "dtype": str(df[col].dtype),
            "num_missing": int(df[col].isnull().sum()),
            "unique_sample": sample_values
        }
    if "Risk_Flag" in df.columns:
        summary["class_balance_sample"] = df["Risk_Flag"].value_counts().to_dict()
    with open(CSV_SUMMARY_PATH, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)
    return summary

csv_summary = create_csv_summary(RAW_CSV_PATH)
print("[INFO] CSV summary created at:", CSV_SUMMARY_PATH)

# -------------------------
# Pydantic schemas (fully parameterized)
# -------------------------
# Branch A
class KnowledgePackage(BaseModel):
    methods: List[str]
    references: List[str]

class PipelinePlan(BaseModel):
    steps: List[str]
    rationale: str

class GeneratedCode(BaseModel):
    preprocess_py: str
    feature_engineering_py: str
    pipeline_plan_yaml: str
    train_stub_py: str

class ReviewReport(BaseModel):
    issues: List[str]
    severity: List[str]
    suggestions: List[str]

# Branch B
class TableData(BaseModel):
    name: str
    columns: List[str]
    rows_preview: List[Dict[str, Any]]    # fully typed

class ChartInfo(BaseModel):
    chart_name: str
    filename: str
    caption: str

class FinalReport(BaseModel):
    title: str
    tables: List[TableData]
    charts: List[ChartInfo]
    suggestions: List[str]
    summary: str

# -------------------------
# Tools
# -------------------------
csv_summary_tool = FileReadTool(file_path=CSV_SUMMARY_PATH)
serper_tool = SerperDevTool() 
scrape_tool = ScrapeWebsiteTool()

# -------------------------
# Agents
# -------------------------
# Branch A (data engineering)
research_agent = Agent(
    role="Research Agent",
    goal="Collect best practices for preprocessing credit risk datasets.",
    backstory="Research-focused data engineer, concise outputs.",
    tools=[serper_tool, scrape_tool, csv_summary_tool],
    verbose=False
)

analyst_agent = Agent(
    role="Analyst Agent",
    goal="Analyze csv_summary and knowledge to produce a PipelinePlan JSON.",
    backstory="Data analyst mapping recommendations to ordered pipeline.",
    tools=[csv_summary_tool],
    verbose=False
)

engineer_agent = Agent(
    role="Engineer Agent",
    goal="Generate reproducible code artifacts (preprocess.py, feature_engineering.py, pipeline yaml, train stub).",
    backstory="Data engineer who writes commented, testable code.",
    tools=[csv_summary_tool],
    verbose=False
)

reviewer_agent = Agent(
    role="Reviewer Agent",
    goal="Produce a short code review report with prioritized fixes.",
    backstory="Senior reviewer focusing on correctness and safety.",
    tools=[csv_summary_tool],
    verbose=False
)

# Branch B (reporting)
suggestion_agent = Agent(
    role="Suggestion Agent",
    goal="Produce concise, prioritized suggestions for data quality improvements.",
    backstory="Data-quality specialist.",
    tools=[csv_summary_tool],
    verbose=False
)

reporting_agent = Agent(
    role="Reporting Agent",
    goal="Produce table summaries and assemble final report combining tables, charts, suggestions.",
    backstory="Tech writer skilled at creating stakeholder-ready reports.",
    tools=[csv_summary_tool],
    verbose=False
)

chart_agent = Agent(
    role="Chart Agent",
    goal="Generate charts (save png) from table summaries and return ChartInfo entries with filenames.",
    backstory="Visualization expert; if execution not allowed, return filenames/instructions.",
    tools=[csv_summary_tool],
    allow_code_execution=False,
    verbose=False
)

# -------------------------
# Tasks (attach output_json where required)
# -------------------------
# Branch A tasks
collect_research = Task(
    description="Collect best-practices for preprocessing credit risk datasets.",
    expected_output="KnowledgePackage JSON",
    output_json=KnowledgePackage,
    output_file=os.path.join(ARTIFACTS_DIR, "knowledge_package.json"),
    agent=research_agent
)

analyze_data = Task(
    description="Produce pipeline plan (PipelinePlan JSON) from csv_summary and knowledge.",
    expected_output="PipelinePlan JSON",
    output_json=PipelinePlan,
    output_file=os.path.join(ARTIFACTS_DIR, "pipeline_plan.json"),
    agent=analyst_agent,
    context=[collect_research]
)

generate_code = Task(
    description="Generate code artifacts from pipeline plan.",
    expected_output="GeneratedCode JSON",
    output_json=GeneratedCode,
    output_file=os.path.join(ARTIFACTS_DIR, "generated_code.json"),
    agent=engineer_agent,
    context=[analyze_data]
)

review_code = Task(
    description="Review generated code artifacts.",
    expected_output="ReviewReport JSON",
    output_json=ReviewReport,
    output_file=os.path.join(ARTIFACTS_DIR, "review_report.json"),
    agent=reviewer_agent,
    context=[generate_code]
)

# Branch B tasks
suggestion_task = Task(
    description="Generate actionable suggestions for dataset quality.",
    expected_output="List of suggestions",
    agent=suggestion_agent
)

table_task = Task(
    description="Create table summaries (TableData) from csv_summary",
    expected_output="TableData JSON",
    output_json=TableData,
    output_file=os.path.join(ARTIFACTS_DIR, "table_data.json"),
    agent=reporting_agent
)

chart_task = Task(
    description="Generate chart images (ChartInfo) from table_data",
    expected_output="ChartInfo JSON",
    output_json=ChartInfo,
    output_file=os.path.join(ARTIFACTS_DIR, "charts_info.json"),
    agent=chart_agent,
    context=[table_task]
)

final_report_task = Task(
    description="Assemble final report combining tables, charts, suggestions into FinalReport JSON.",
    expected_output="FinalReport JSON",
    output_json=FinalReport,
    output_file=os.path.join(ARTIFACTS_DIR, "final_report.json"),
    agent=reporting_agent,
    context=[table_task, chart_task, suggestion_task]
)

# -------------------------
# Crew
# -------------------------
crew = Crew(
    agents=[
        research_agent, analyst_agent, engineer_agent, reviewer_agent,
        suggestion_agent, reporting_agent, chart_agent
    ],
    tasks=[
        collect_research, analyze_data, generate_code, review_code,
        suggestion_task, table_task, chart_task, final_report_task
    ],
    verbose=True
)

# -------------------------
# Run crew (safe)
# -------------------------
print("[INFO] Running Crew (may call LLMs/tools)...")
try:
    result = crew.kickoff()
except Exception as e:
    # save error and re-raise so you can inspect locally
    print("[ERROR] Crew.kickoff() failed:", e)
    with open(CREW_RAW_JSON, "w", encoding="utf-8") as f:
        f.write("Crew kickoff failed with exception:\n")
        f.write(repr(e))
    raise

# try saving raw result (best-effort)
try:
    raw = getattr(result, "raw", result)
    with open(CREW_RAW_JSON, "w", encoding="utf-8") as f:
        json.dump(raw, f, indent=2, ensure_ascii=False)
except Exception:
    with open(CREW_RAW_JSON, "w", encoding="utf-8") as f:
        f.write(str(result))

print("[INFO] Crew finished. Short preview (truncated):")
_preview = str(getattr(result, "raw", result))[:1500]
print(_preview + ("...[truncated]" if len(str(getattr(result, "raw", result)))>1500 else ""))

# -------------------------
# Assemble unified Markdown report (merge branch A + branch B)
# -------------------------
def safe_load_json(path: str):
    if not os.path.exists(path):
        return None
    try:
        return json.load(open(path, encoding="utf-8"))
    except Exception:
        try:
            with open(path, "r", encoding="utf-8") as f:
                return json.loads(f.read())
        except Exception:
            return None

def assemble_markdown():
    md_lines: List[str] = []
    md_lines.append("# Multi-Agent Data Engineering & Reporting Summary\n")
    md_lines.append(f"_Generated: {pd.Timestamp.now()}_\n\n")

    # 1) Branch A: pipeline plan, code preview, review findings
    md_lines.append("## A. Data Engineering (Pipeline & Code)\n")

    kp = safe_load_json(os.path.join(ARTIFACTS_DIR, "knowledge_package.json"))
    if kp:
        md_lines.append("### Knowledge package (summary)\n")
        for m in kp.get("methods", kp.get("methods", [])):
            md_lines.append(f"- {m}")
        md_lines.append("")

    pp = safe_load_json(os.path.join(ARTIFACTS_DIR, "pipeline_plan.json"))
    if pp:
        try:
            plan = PipelinePlan(**pp)
            md_lines.append("### Pipeline Plan\n")
            for i, s in enumerate(plan.steps, 1):
                md_lines.append(f"{i}. {s}")
            md_lines.append("\n**Rationale:**\n")
            md_lines.append(plan.rationale + "\n")
        except Exception:
            md_lines.append("*(pipeline_plan.json exists but could not parse to schema)*\n")

    gc = safe_load_json(os.path.join(ARTIFACTS_DIR, "generated_code.json"))
    if gc:
        md_lines.append("### Generated Code (previews)\n")
        # show small preview for each file if present
        for key in ["preprocess_py", "feature_engineering_py", "pipeline_plan_yaml", "train_stub_py"]:
            if key in gc:
                snippet = gc.get(key, "")[:1000]
                md_lines.append(f"#### {key}\n```python\n{snippet}\n```\n")
    else:
        md_lines.append("*(No generated_code.json found)*\n")

    rr = safe_load_json(os.path.join(ARTIFACTS_DIR, "review_report.json"))
    if rr:
        try:
            review = ReviewReport(**rr)
            md_lines.append("### Code Review Summary\n")
            for issue, sev, sug in zip(review.issues, review.severity, review.suggestions):
                md_lines.append(f"- **{sev}**: {issue}\n  - Suggestion: {sug}")
            md_lines.append("")
        except Exception:
            md_lines.append("*(review_report.json exists but could not parse)*\n")
    else:
        md_lines.append("*(No review_report.json found)*\n")

    # 2) Branch B: tables, charts, suggestions, final_report summary
    md_lines.append("\n\n## B. Reporting (Tables, Charts, Suggestions)\n")
    tables = safe_load_json(os.path.join(ARTIFACTS_DIR, "table_data.json"))
    if tables:
        # tables might be a single object or list
        tlist = tables if isinstance(tables, list) else [tables]
        for t in tlist:
            try:
                t_obj = TableData(**t)
                md_lines.append(f"### Table: {t_obj.name}\n")
                md_lines.append("| " + " | ".join(t_obj.columns) + " |")
                md_lines.append("|" + " --- |" * len(t_obj.columns))
                for row in t_obj.rows_preview:
                    md_lines.append("| " + " | ".join(str(row.get(c, "")) for c in t_obj.columns) + " |")
                md_lines.append("")
            except Exception:
                md_lines.append(f"*(Could not parse table object: {t})*\n")
    else:
        md_lines.append("*(No table_data.json found)*\n")

    charts = safe_load_json(os.path.join(ARTIFACTS_DIR, "charts_info.json"))
    if charts:
        clist = charts if isinstance(charts, list) else [charts]
        md_lines.append("### Charts\n")
        for c in clist:
            try:
                c_obj = ChartInfo(**c)
                md_lines.append(f"#### {c_obj.chart_name}\n")
                if os.path.exists(c_obj.filename):
                    # use relative path for markdown
                    rel = os.path.relpath(c_obj.filename, ARTIFACTS_DIR)
                    md_lines.append(f"![{c_obj.caption}]({rel})\n")
                else:
                    md_lines.append(f"*(Chart file not found: {c_obj.filename})*\n")
            except Exception:
                md_lines.append(f"*(Could not parse chart entry: {c})*\n")
    else:
        md_lines.append("*(No charts_info.json found)*\n")

    suggestions = safe_load_json(os.path.join(ARTIFACTS_DIR, "suggestion_task.json")) or safe_load_json(os.path.join(ARTIFACTS_DIR, "suggestions.json"))
    if suggestions:
        md_lines.append("### Suggestions\n")
        if isinstance(suggestions, list):
            for s in suggestions:
                md_lines.append(f"- {s}")
        elif isinstance(suggestions, dict) and "suggestions" in suggestions:
            for s in suggestions["suggestions"]:
                md_lines.append(f"- {s}")
        else:
            md_lines.append(str(suggestions))
        md_lines.append("")
    else:
        md_lines.append("*(No suggestions found)*\n")

    # try final_report.json (if produced)
    fr = safe_load_json(os.path.join(ARTIFACTS_DIR, "final_report.json"))
    if fr:
        try:
            fobj = FinalReport(**fr)
            md_lines.append("\n## Final Report Summary (from reporting crew)\n")
            md_lines.append(fobj.summary + "\n")
        except Exception:
            md_lines.append("*(final_report.json exists but could not parse)*\n")

    # combine and write
    md_text = "\n".join(md_lines)
    with open(FINAL_REPORT_MD, "w", encoding="utf-8") as f:
        f.write(md_text)
    print("[INFO] Final markdown assembled:", FINAL_REPORT_MD)
    # display in notebook if possible
    try:
        display(Markdown(md_text))
    except Exception:
        pass

# assemble now
assemble_markdown()


2025-09-23 10:01:30,473 - 140327692086144 - __init__.py-__init__:537 - WARNING: Overriding of current TracerProvider is not allowed


[INFO] CSV summary created at: artifacts/csv_summary.json
[INFO] Running Crew (may call LLMs/tools)...
# Agent: Research Agent
## Task: Collect best-practices for preprocessing credit risk datasets.


# Agent: Research Agent
## Thought: I need to gather information on best practices for preprocessing credit risk datasets. I will search for relevant resources online.
## Using tool: Search the internet with Serper
## Tool Input: 
"{\"search_query\": \"best practices for preprocessing credit risk datasets\"}"
## Tool Output: 
{'searchParameters': {'q': 'best practices for preprocessing credit risk datasets', 'type': 'search', 'num': 10, 'engine': 'google'}, 'organic': [{'title': 'Comprehensive Data Preparation for Machine Learning with Credit ...', 'link': 'https://www.linkedin.com/pulse/comprehensive-data-preparation-machine-learning-risk-ravichandran-88edc', 'snippet': 'Preprocessing involves steps like handling missing values, removing outliers, standardizing formats, and encoding data



# Agent: Research Agent
## Thought: I will gather more references to further solidify the best practices for preprocessing credit risk datasets.
## Using tool: Read website content
## Tool Input: 
"{\"website_url\": \"https://www.leewayhertz.com/build-credit-risk-models-using-machine-learning/\"}"
## Tool Output: 

A guide to credit risk modeling using machine learning
The Hackett Group Announces Strategic Acquisition of Leading Gen AI Development Firm LeewayHertz
Read more
Toggle Toggle AI PRODUCTS
Enterprise GenAI Platform
AI Copilot for Sales
AI Research Solution for Due Diligence
GenAI Platform for Healthcare
AI Customer Service Agent
GenAI Platform for Finance
GenAI Platform for Manufacturing
GenAI Platform for Logistics
SERVICES
Generative AI
Generative AI Development
Generative AI Integration Services
Generative AI Consulting Company
Hire Generative AI Engineers
AI Agent Development
AI Copilot Development
AI Marketing Agent Development
Hire Prompt Engineers
Adaptive AI Develop



# Agent: Research Agent
## Thought: I will now read another relevant article to extract further methods and references on preprocessing credit risk datasets.
## Using tool: Read website content
## Tool Input: 
"{\"website_url\": \"https://coralogix.com/ai-blog/credit-risk-modeling-importance-model-types-and-10-best-practices/\"}"
## Tool Output: 

Credit Risk Modeling: Importance, Types & 10 Best Practices
Skip to content
Platform
Platform overview
Platform overview
Platform capabilities
Remote, index-free querying
Infinite retention
DataPrime engine
In-stream analysis & alerting
Cross-stack dashboards
Cost optimization tool
Zero instrumentation
System
APM
Service catalog
DB monitoring
Service map
Serverless APM
Continuous profiling
Dependencies
SLO management
Real user monitoring
Error tracking
Session replay
Core web vitals
Measurements
Network monitoring
Versions
Infrastructure monitoring
Infrastructure explorer
Log analytics
Log templating
Parsing rules
Data enrichment
AI
AI disc



# Agent: Research Agent
## Final Answer: 
{
  "methods": [
    "Data Quality: Ensure accuracy, completeness, and relevance of data used for training models.",
    "Regularization: Utilize L1 and L2 techniques to avoid overfitting in machine learning models.",
    "Model Validation: Conduct rigorous testing on separate datasets to verify model assumptions and effectiveness.",
    "Model Transparency: Make models interpretable to build trust among stakeholders.",
    "Model Reassessment: Continuously update models to adapt to changing data and economic conditions.",
    "Model Documentation: Thoroughly document methodologies, inputs, outputs, and limitations.",
    "Model Governance: Establish clear roles, procedures, and tracking systems for model changes.",
    "Data Privacy: Protect sensitive data and ensure compliance with regulations.",
    "Model Diversity: Implement multiple models to reduce reliance on a single solution.",
    "Model Explainability: Ensure models are understand



# Agent: Reviewer Agent
## Thought: I need to analyze the provided Python scripts to identify any potential issues, verify correctness, and ensure safety in the code.
## Using tool: Read a file's content
## Tool Input: 
"{\"file_path\": \"artifacts/csv_summary.json\"}"
## Tool Output: 
{
  "sample_rows": 500,
  "num_columns": 13,
  "columns": {
    "Id": {
      "dtype": "int64",
      "num_missing": 0,
      "unique_sample": [
        1,
        2,
        3,
        4,
        5
      ]
    },
    "Income": {
      "dtype": "int64",
      "num_missing": 0,
      "unique_sample": [
        1303834,
        7574516,
        3991815,
        6256451,
        5768871
      ]
    },
    "Age": {
      "dtype": "int64",
      "num_missing": 0,
      "unique_sample": [
        23,
        40,
        66,
        41,
        47
      ]
    },
    "Experience": {
      "dtype": "int64",
      "num_missing": 0,
      "unique_sample": [
        3,
        10,
        4,
        2,
        11




# Agent: Reporting Agent
## Final Answer: 
{
  "name": "TableData",
  "columns": [
    "Id",
    "Income",
    "Age",
    "Experience",
    "Married/Single",
    "House_Ownership",
    "Car_Ownership",
    "Profession",
    "CITY",
    "STATE",
    "CURRENT_JOB_YRS",
    "CURRENT_HOUSE_YRS",
    "Risk_Flag"
  ],
  "rows_preview": [
    {
      "Id": 1,
      "Income": 1303834,
      "Age": 23,
      "Experience": 3,
      "Married/Single": "single",
      "House_Ownership": "rented",
      "Car_Ownership": "no",
      "Profession": "Mechanical_engineer",
      "CITY": "Rewa",
      "STATE": "Madhya_Pradesh",
      "CURRENT_JOB_YRS": 3,
      "CURRENT_HOUSE_YRS": 13,
      "Risk_Flag": 0
    },
    {
      "Id": 2,
      "Income": 7574516,
      "Age": 40,
      "Experience": 10,
      "Married/Single": "married",
      "House_Ownership": "owned",
      "Car_Ownership": "yes",
      "Profession": "Software_Developer",
      "CITY": "Parbhani",
      "STATE": "Maharashtra",
      "CURR



# Agent: Reporting Agent
## Final Answer: 
{
  "title": "Final Report on Income and Demographic Analysis",
  "tables": [{
    "name": "TableData",
    "columns": [
      "Id",
      "Income",
      "Age",
      "Experience",
      "Married/Single",
      "House_Ownership",
      "Car_Ownership",
      "Profession",
      "CITY",
      "STATE",
      "CURRENT_JOB_YRS",
      "CURRENT_HOUSE_YRS",
      "Risk_Flag"
    ],
    "rows_preview": [
      {
        "Id": 1,
        "Income": 1303834,
        "Age": 23,
        "Experience": 3,
        "Married/Single": "single",
        "House_Ownership": "rented",
        "Car_Ownership": "no",
        "Profession": "Mechanical_engineer",
        "CITY": "Rewa",
        "STATE": "Madhya_Pradesh",
        "CURRENT_JOB_YRS": 3,
        "CURRENT_HOUSE_YRS": 13,
        "Risk_Flag": 0
      },
      {
        "Id": 2,
        "Income": 7574516,
        "Age": 40,
        "Experience": 10,
        "Married/Single": "married",
        "House_Owner

# Multi-Agent Data Engineering & Reporting Summary

_Generated: 2025-09-23 10:03:25.095974_


## A. Data Engineering (Pipeline & Code)

### Knowledge package (summary)

- Data Quality: Ensure accuracy, completeness, and relevance of data used for training models.
- Regularization: Utilize L1 and L2 techniques to avoid overfitting in machine learning models.
- Model Validation: Conduct rigorous testing on separate datasets to verify model assumptions and effectiveness.
- Model Transparency: Make models interpretable to build trust among stakeholders.
- Model Reassessment: Continuously update models to adapt to changing data and economic conditions.
- Model Documentation: Thoroughly document methodologies, inputs, outputs, and limitations.
- Model Governance: Establish clear roles, procedures, and tracking systems for model changes.
- Data Privacy: Protect sensitive data and ensure compliance with regulations.
- Model Diversity: Implement multiple models to reduce reliance on a single solution.
- Model Explainability: Ensure models are understandable to comply with regulations and community standards.

### Pipeline Plan

1. Data Quality Check
2. Data Preprocessing
3. Feature Engineering
4. Model Selection
5. Model Training
6. Model Validation
7. Model Reassessing
8. Model Documentation
9. Model Governance and Transparency
10. Model Explainability

**Rationale:**

The steps include ensuring data quality, preparing and engineering data, selecting and training models while focusing on validation and governance, which are critical for building reliable predictive models in credit risk assessment.

### Generated Code (previews)

#### preprocess_py
```python
# preprocess.py

import pandas as pd


def load_data(file_path):
    """Load data from a CSV file."""
    return pd.read_csv(file_path)


def data_quality_check(df):
    """Perform data quality checks."""
    assert df.isnull().sum().sum() == 0, "Data contains missing values"
    assert df.shape[1] == 13, "Unexpected number of columns"


def preprocess_data(df):
    """Preprocess the data by encoding categorical variables."""
    df['Married/Single'] = df['Married/Single'].map({'single': 0, 'married': 1})
    df['House_Ownership'] = df['House_Ownership'].map({'rented': 0, 'norent_noown': 1, 'owned': 2})
    df['Car_Ownership'] = df['Car_Ownership'].map({'no': 0, 'yes': 1})
    return df


if __name__ == '__main__':
    df = load_data('data/credit_risk.csv')
    data_quality_check(df)
    processed_df = preprocess_data(df)

```

#### feature_engineering_py
```python
# feature_engineering.py

import pandas as pd


def feature_engineering(df):
    """Create new features from existing data."""
    df['Income_to_Age_Ratio'] = df['Income'] / df['Age']
    df['Experience_to_Job_Years_Ratio'] = df['Experience'] / df['CURRENT_JOB_YRS']
    return df


if __name__ == '__main__':
    df = pd.read_csv('data/processed_credit_risk.csv')
    enhanced_df = feature_engineering(df)

```

#### pipeline_plan_yaml
```python
steps:
  - name: Data Quality Check
    script: preprocess.py
  - name: Data Preprocessing
    script: preprocess.py
  - name: Feature Engineering
    script: feature_engineering.py
  - name: Model Selection
    script: model_selection.py
  - name: Model Training
    script: model_training.py
  - name: Model Validation
    script: model_validation.py
  - name: Model Reassessing
    script: model_reassessing.py
  - name: Model Documentation
    script: model_documentation.py
  - name: Model Governance and Transparency
    script: model_governance.py
  - name: Model Explainability
    script: model_explainability.py

```

#### train_stub_py
```python
# train_stub.py

import pandas as pd
from model_selection import select_model
from model_training import train_model

def main():
    df = pd.read_csv('data/feature_engineered_data.csv')
    model = select_model(df)
    train_model(model, df)

if __name__ == '__main__':
    main()

```

### Code Review Summary

- **high**: Data quality check using assert statements may raise AssertionError without clear error handling.
  - Suggestion: Implement proper error handling instead of assert statements to make issues more transparent and manageable.
- **medium**: The mapping in preprocess_data assumes the values in the columns are well-defined, which can lead to errors if unexpected values are present.
  - Suggestion: Add value validation before performing mapping to prevent runtime errors.
- **high**: Lack of validation for division by zero in feature_engineering can cause runtime errors.
  - Suggestion: Check for zero values before performing division to avoid potential ZeroDivisionError.
- **medium**: The main block in all scripts assumes the presence of specific CSV files without checks, which may cause FileNotFoundError.
  - Suggestion: Add checks to ensure the required CSV files exist before attempting to load them.



## B. Reporting (Tables, Charts, Suggestions)

### Table: TableData

| Id | Income | Age | Experience | Married/Single | House_Ownership | Car_Ownership | Profession | CITY | STATE | CURRENT_JOB_YRS | CURRENT_HOUSE_YRS | Risk_Flag |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | 1303834 | 23 | 3 | single | rented | no | Mechanical_engineer | Rewa | Madhya_Pradesh | 3 | 13 | 0 |
| 2 | 7574516 | 40 | 10 | married | owned | yes | Software_Developer | Parbhani | Maharashtra | 9 | 10 | 1 |
| 3 | 3991815 | 66 | 4 | single | norent_noown | no | Technical_writer | Alappuzha | Kerala | 4 | 12 | 0 |
| 4 | 6256451 | 41 | 2 | married | owned | yes | Civil_servant | Bhubaneswar | Odisha | 2 | 14 | 1 |
| 5 | 5768871 | 47 | 11 | single | rented | no | Librarian | Tiruchirappalli[10] | Tamil_Nadu | 0 | 11 | 0 |

### Charts

#### Income Distribution

*(Chart file not found: income_distribution.png)*

*(No suggestions found)*


## Final Report Summary (from reporting crew)

This report provides a comprehensive overview of income and demographic factors, including age, experience, marital status, house ownership, and car ownership among individuals in the dataset. It highlights the need for error handling, data validation, and class balance assessment for further data analysis.


## Flow Chart

In [10]:
# =========================================
# Credit Risk Pipeline Flow
# =========================================
import os
from crewai import Flow
from crewai.flow.flow import listen, start, and_
from IPython.display import IFrame

class CreditRiskPipeline(Flow):
    # --- ROOT: Dataset Input ---
    @start()
    def load_dataset(self):
        return {"status": "done", "task": "load_dataset"}

    # --- Branch A: Data Engineering ---
    @listen(load_dataset)
    def collect_research(self, state):
        return {"status": "done", "task": "collect_research"}

    @listen(collect_research)
    def analyze_data(self, state):
        return {"status": "done", "task": "analyze_data"}

    @listen(analyze_data)
    def generate_code(self, state):
        return {"status": "done", "task": "generate_code"}

    @listen(generate_code)
    def review_code(self, state):
        return {"status": "done", "task": "review_code"}

    # --- Branch B: Reporting ---
    @listen(load_dataset)
    def suggestion_task(self, state):
        return {"status": "done", "task": "suggestion_task"}

    @listen(load_dataset)
    def table_task(self, state):
        return {"status": "done", "task": "table_task"}

    @listen(table_task)
    def chart_task(self, state):
        return {"status": "done", "task": "chart_task"}

    # cần tất cả suggestion + table + chart
    @listen(and_(suggestion_task, table_task, chart_task))
    def final_report_task(self, state):
        return {"status": "done", "task": "final_report_task"}

    # --- Merge 2 nhánh ---
    @listen(and_(review_code, final_report_task))
    def assemble_markdown(self, state):
        return {"status": "done", "task": "assemble_markdown"}


# =========================================
# Khởi tạo flow & vẽ sơ đồ
# =========================================
flow = CreditRiskPipeline()

# Xuất sơ đồ flowchart ra file HTML
flow.plot("./crewai_flow.html")

# Hiển thị trong notebook
IFrame(src='./crewai_flow.html', width='150%', height=600)


Plot saved as ./crewai_flow.html.html
